In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [22]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [5]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)


26

In [8]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7032.50it/s]


In [9]:
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [ ]:
## AI Agent = tools , llm , prompt

In [18]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrive the relevant data of the pdf document and these pdf documents have details about medical reports.
    """
    docs = vector_store.similarity_search(query=query , k = 4)
    context = ""
    for doc in docs:
        context = doc.page_content + "\n\n"
        
    return context

In [21]:
llm = ChatGroq(model="openai/gpt-oss-20b")


In [23]:
system_prompt = """
    You are a helpful assistant that answers questions using retrieved context.
    ALWAYS use the retriever tool for questions requiring external knowledge
"""

In [24]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)


In [26]:
query = "What is the name of patient and what is the name of doctors?"
res = agent.invoke({"messages":{"role":"user","content":query}})
res["messages"][-1].content

'**Patient**  \n- **Ms.\u202fNikita\u202fChudhary**\n\n**Doctors involved in the report**\n\n| Role | Doctor | Institution |\n|------|--------|-------------|\n| Referring/Reporting doctor | **Dr\u202fNitin\u202fNahar** | (listed in the header) |\n| Pathology Consultant | **Dr\u202fSunanda** | NRL – Dr\u202fLal\u202fPathLabs Ltd |\n| Chief of Laboratory | **Dr\u202fSwapnil\u202fGupta** | Dr\u202fLal\u202fPathLabs Ltd |\n| Chief of Laboratory | **Dr\u202fKiran\u202fBhargava\u202fPathak** | Dr\u202fLal\u202fPathLabs Ltd |\n\nThese names appear throughout the report (e.g., “Ms.\u202fNikita\u202fChudhary” for the patient, and the various “Dr.” entries for the pathology and laboratory team).'